# Predicting Sales from Campaign Data

End-to-end reproducible notebook prepared by **Mohit Bhatnagar**.

## 1. Load data

In [ ]:
import pandas as pd
train = pd.read_csv('messy_train_data.csv', dtype=str)
test = pd.read_csv('messy_test_data.csv', dtype=str)
print(train.shape, test.shape)
train.head()

## 2. Reusable cleaning and prediction module

In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REQUIRED_FEATURE_COLUMNS=["Followers","EngagementRate (%)","AdSpend (GBP)","ContentQuality"]

def _parse_numeric(series):
    cleaned=(series.astype("string").str.replace(r"[£,%\s,]","",regex=True).replace({"":pd.NA,"nan":pd.NA,"None":pd.NA,"N/A":pd.NA}))
    return pd.to_numeric(cleaned,errors="coerce")

class CampaignCleaner(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): self._validate_columns(X); return self
    def transform(self,X):
        self._validate_columns(X); out=pd.DataFrame(index=X.index)
        f=_parse_numeric(X["Followers"]); f=f.mask(f<1000,f*1000); f=f.mask(f>1_000_000,f/10_000)
        out["Followers"]=f
        out["EngagementRate"]=_parse_numeric(X["EngagementRate (%)"]).abs()
        s=_parse_numeric(X["AdSpend (GBP)"]).abs(); s=s.mask(s>100_000,s/100_000)
        out["AdSpend"]=s; out["ContentQuality"]=_parse_numeric(X["ContentQuality"])
        return out.astype(float)
    @staticmethod
    def _validate_columns(X):
        if not isinstance(X,pd.DataFrame): raise TypeError("CampaignCleaner expects a pandas DataFrame.")
        missing=[c for c in REQUIRED_FEATURE_COLUMNS if c not in X.columns]
        if missing: raise ValueError(f"Missing required columns: {missing}")

def build_model(alpha=1.0):
    return Pipeline([("cleaner",CampaignCleaner()),("imputer",SimpleImputer(strategy="median",add_indicator=True)),("scaler",StandardScaler()),("model",Ridge(alpha=alpha))])

def fit_model(train_df,alpha=1.0):
    if "Sales (Units)" not in train_df.columns: raise ValueError("Training data must contain 'Sales (Units)'.")
    y=_parse_numeric(train_df["Sales (Units)"])
    if y.isna().any(): raise ValueError("Target contains missing or non-numeric values.")
    return build_model(alpha).fit(train_df,y)

def predict_sales(model,campaign_df):
    p=np.clip(model.predict(campaign_df),0,None)
    out=pd.DataFrame({"Predicted_Sales_Units":np.rint(p).astype(int)},index=campaign_df.index)
    if "ID" in campaign_df.columns: out.insert(0,"ID",pd.to_numeric(campaign_df["ID"],errors="coerce").astype("Int64"))
    return out.reset_index(drop=True)

def audit_data_quality(df):
    CampaignCleaner._validate_columns(df); f=_parse_numeric(df["Followers"]);e=_parse_numeric(df["EngagementRate (%)"]);s=_parse_numeric(df["AdSpend (GBP)"])
    return {"rows":int(len(df)),"missing_followers":int(df["Followers"].isna().sum()),"small_scale_followers":int((f<1000).sum()),"inflated_followers":int((f>1_000_000).sum()),"missing_engagement":int(df["EngagementRate (%)"].isna().sum()),"percentage_strings":int(df["EngagementRate (%)"].fillna("").astype(str).str.contains("%",regex=False).sum()),"negative_engagement":int((e<0).sum()),"missing_spend":int(df["AdSpend (GBP)"].isna().sum()),"currency_strings":int(df["AdSpend (GBP)"].fillna("").astype(str).str.contains("£",regex=False).sum()),"negative_spend":int((s<0).sum()),"inflated_spend":int((s>100_000).sum()),"missing_content_quality":int(df["ContentQuality"].isna().sum())}


## 3. Fit the selected Ridge pipeline and predict the test set

In [ ]:
model = fit_model(train, alpha=1.0)
predictions = predict_sales(model, test)
predictions.to_csv('test_sales_predictions.csv', index=False)
predictions.head()

## Final validated result

Tuned Ridge Regression (`alpha=1`) was selected using shuffled five-fold cross-validation.

| Metric | Result |
|---|---:|
| RMSE | 2,000.36 |
| MAE | 1,592.73 |
| R-squared | 0.4925 |

Tuned XGBoost produced RMSE 2,016.98, so the simpler Ridge model was retained.

## 4. Data-quality audit

In [ ]:
pd.Series(audit_data_quality(train), name='Training count').to_frame()

## 5. Interpretation

Approximate fitted effects, holding the other measured attributes constant:

- +10,000 followers -> +504 predicted units
- +1 percentage point engagement -> +180 units
- +GBP 1,000 ad spend -> +789 units
- +1 content-quality point -> +189 units

These are predictive associations, not causal effects.